In [16]:
import os
import pandas as pd

path = "7_hospitales_clientes_ncd"
output_path = "data"

os.makedirs(output_path, exist_ok=True)

# 1. Listar los archivos CSV
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]

if not csv_files:
    print("[ALERTA] No se encontraron archivos CSV.")
    exit()

# ==========================================
# PASO 1: CREAR EL ESQUEMA DE COLUMNAS MAESTRO
# ==========================================
print("[INFO] Analizando categorías globales para fijar el espacio de características...")
todas_las_columnas_encoded = set()

# Columnas que eliminaremos estrictamente de todo el pipeline
columnas_a_remover = ['is_premature_ncd', 'hospital_cliente', 'anio_fall']

for file in csv_files:
    df_temp = pd.read_csv(os.path.join(path, file))
    
    # Eliminamos las variables indicadas antes de simular el encoding
    X_temp = df_temp.drop(columns=columnas_a_remover, errors='ignore')
    
    # Generamos el One-Hot para capturar los nombres de las columnas resultantes
    X_enc_temp = pd.get_dummies(X_temp, drop_first=True, dtype=int)
    todas_las_columnas_encoded.update(X_enc_temp.columns)

# Convertimos a una lista ordenada para que el orden de los tensores coincida en los nodos
columnas_maestras = sorted(list(todas_las_columnas_encoded))
print(f"[INFO] Espacio de características unificado: {len(columnas_maestras)} columnas (sin 'anio_fall' ni 'hospital_cliente').\n")


# ==========================================
# PASO 2: PROCESAR CADA CSV DE FORMA INDEPENDIENTE
# ==========================================
for file in csv_files:
    file_path = os.path.join(path, file)
    print(f"[PROCESANDO SEPARADO] -> {file}")
    
    # Leer el archivo de forma aislada
    df = pd.read_csv(file_path)
    
    # 1. Tratar y aislar el label (is_premature_ncd se queda como columna 0)
    df['is_premature_ncd'] = df['is_premature_ncd'].astype(int)
    labels = df['is_premature_ncd']
    
    # 2. Aislar variables X quitando el label, el metadato del nodo y la columna de año
    X_raw = df.drop(columns=columnas_a_remover, errors='ignore')
    
    # 3. Aplicar One-Hot de forma local
    X_encoded = pd.get_dummies(X_raw, drop_first=True, dtype=int)
    
    # 4. ALINEACIÓN ESTRUCTURAL: Reindexar usando el catálogo maestro
    # Las columnas ausentes en este nodo se inicializan con 0 automáticamente
    X_aligned = X_encoded.reindex(columns=columnas_maestras, fill_value=0)
    
    # 5. Reconstruir el DataFrame final (Solo Label + Características alineadas)
    df_nodo_final = pd.concat([labels, X_aligned], axis=1)
    
    # Guardar el archivo limpio en la carpeta processed
    output_file_path = os.path.join(output_path, file)
    df_nodo_final.to_csv(output_file_path, index=False)
    print(f"    [GUARDADO] Filas: {df_nodo_final.shape[0]} | Columnas Totales: {df_nodo_final.shape[1]}")

print(f"\n[PROCESO TERMINADO] Todos los CSVs guardados en '{output_path}' listos para las tensores de PyTorch.")

[INFO] Analizando categorías globales para fijar el espacio de características...
[INFO] Espacio de características unificado: 79 columnas (sin 'anio_fall' ni 'hospital_cliente').

[PROCESANDO SEPARADO] -> Hospital_5.csv
    [GUARDADO] Filas: 1868 | Columnas Totales: 80
[PROCESANDO SEPARADO] -> Hospital_6.csv
    [GUARDADO] Filas: 1370 | Columnas Totales: 80
[PROCESANDO SEPARADO] -> Hospital_1.csv
    [GUARDADO] Filas: 5524 | Columnas Totales: 80
[PROCESANDO SEPARADO] -> Hospital_2.csv
    [GUARDADO] Filas: 3340 | Columnas Totales: 80
[PROCESANDO SEPARADO] -> Hospital_7.csv
    [GUARDADO] Filas: 881 | Columnas Totales: 80
[PROCESANDO SEPARADO] -> Hospital_3.csv
    [GUARDADO] Filas: 2724 | Columnas Totales: 80
[PROCESANDO SEPARADO] -> Hospital_4.csv
    [GUARDADO] Filas: 2294 | Columnas Totales: 80

[PROCESO TERMINADO] Todos los CSVs guardados en 'data' listos para las tensores de PyTorch.
